In [1]:
import os
import cv2
import numpy as np

In [2]:
data = []
labels = []

In [3]:
for label in range(10):
    folder = f"./dataset/{label}"
    for img_name in os.listdir(folder):
        img_path = os.path.join(folder,img_name)
        img = cv2.imread(img_path,cv2.IMREAD_GRAYSCALE)
        data.append(img)
        labels.append(label)

In [4]:
x = np.array(data)
y = np.array(labels)

In [5]:
x = x/255.0
x = x.reshape(-1, 56, 56, 1)

In [6]:
from sklearn.model_selection import train_test_split

In [8]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [10]:
model = Sequential([
    
    # Conv Block 1
    Conv2D(32, (3,3), activation='relu', input_shape=(56,56,1)),
    MaxPooling2D(2,2),
    
    # Conv Block 2
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    # Conv Block 3
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    # Flatten
    Flatten(),

    # Dense Layers
    Dense(128, activation='relu'),
    Dropout(0.5),

    # Output (for digits 0–9)
    Dense(10, activation='softmax')
])

In [11]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [12]:
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

In [13]:
datagen.fit(x)

In [14]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [15]:
history = model.fit(
    datagen.flow(x_train, y_train, batch_size=16),
    epochs=20,
    validation_data=(x_test, y_test)
)

ValueError: Asked to retrieve element 0, but the Sequence has length 0

In [ ]:
loss, acc = model.evaluate(x_test, y_test)
print("Accuracy:", acc)

In [ ]:
img = cv2.imread("test.png", cv2.IMREAD_GRAYSCALE)
img = img / 255.0
img = img.reshape(1, 56, 56, 1)

prediction = model.predict(img)
print("Digit:", np.argmax(prediction))

In [ ]:
model.save("digit_model.h5")